# Class 4: Visualization and pandas
28 October 2026 · DM587

Run the cells from top to bottom. Keep odense-weather.csv beside this notebook. Packages: numpy, matplotlib, pandas; statsmodels is needed for the final forecasting section.

We begin with function plots, introduce pandas tables, then explore the weather record near Odense (2006–2025). The final example uses 365 daily observations to predict the following 10 daily mean temperatures. Monthly summaries earlier in the notebook are for exploration, not the forecast targets. All weather values come from the supplied ERA5 reanalysis extract; no weather values are simulated.


## 1. Matplotlib
### Functions
Start with familiar functions: choose a grid of x-values, evaluate each function on the grid and connect the sampled points. `np.linspace(0, 2 * np.pi, 200)` includes 200 evenly spaced points, including both endpoints. Increase or decrease the number of points to see how sampling affects the curve.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
x = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(x, np.sin(x), label="sin(x)", color="#006D77")
ax.plot(x, np.cos(x), label="cos(x)", color="#D56A35")
ax.set_xlabel("x (radians)")
ax.set_ylabel("Function value")
ax.set_title("Sine and cosine")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


## 2. Weather data and pandas
### Questions
How does temperature change through the year? Which temperatures are common or unusual? Are daily minimum and maximum temperatures related? Can past temperature data help predict future temperatures?

We first load and understand the table. These are public ERA5 gridded reanalysis estimates near Odense, not data from a named weather station. See DATA-SOURCE.md for the extraction, units, grid location and CC BY 4.0 licence.

### DataFrame and index
A DataFrame is a table with named columns and labelled rows; a Series is one labelled column. Read the CSV once, inspect its five columns, then use its date column as the index. `head(3)` selects the first three rows; `iloc[0:3, :]` selects those same positions and all columns. The slice stop is excluded. `parse_dates` turns date text into calendar dates.

In [ ]:
data = pd.read_csv("odense-weather.csv", parse_dates=["date"])
print("Columns:", data.columns.tolist())
print("First three complete rows:")
print(data.head(3).to_string(index=False))
print("Same rows with iloc:")
print(data.iloc[0:3, :].to_string(index=False))
weather = data.set_index("date").sort_index()
print("Date-indexed table:")
print(weather.head(3).to_string())
print("Shape:", weather.shape)
print("Types:", weather.dtypes.to_dict())


### One or several columns
Single brackets with a string return a Series. A list of column names returns a DataFrame, even if that list contains only one name. Both keep the date labels.

In [ ]:
temperature = weather["temp_mean"]
one_column = weather[["temp_mean"]]
two_columns = weather[["temp_min", "temp_max"]]
print("Single name:", type(temperature).__name__, temperature.shape)
print("List of one name:", type(one_column).__name__, one_column.shape)
print("Two columns:", two_columns.shape)

### Labels and positions
`loc` uses labels: a date slice includes both stated endpoints. `iloc` uses integer positions and its stop is excluded, as in ordinary Python slicing. Here `loc` selects dates and columns, while `iloc[:3]` selects the first three rows. These operations do not reset the index.

In [ ]:
print("By date labels:")
print(weather.loc["2024-01-01":"2024-01-03", ["temp_mean", "rain_mm"]].to_string())
print("By positions:")
print(weather.iloc[:3].to_string())

### Filter rows
The question is: which days in July 2024 had at least 5 mm of rain? Each condition makes a boolean Series. Combine elementwise conditions with `&` and parentheses, not Python's scalar `and`. Use `loc` to select rows and columns together.

In [ ]:
in_july = weather.loc["2024-07-01":"2024-07-31"]
wet_days = in_july.loc[(in_july["rain_mm"] >= 5), ["temp_mean", "rain_mm"]]
print(wet_days.to_string())
print("Wet days:", len(wet_days))
print("Warm and wet in July:", len(in_july.loc[(in_july["temp_mean"] > 18) & (in_july["rain_mm"] >= 5)]))

### Derived columns
Subtract whole columns; do not write a Python loop over rows for this calculation. Pandas aligns Series by their index labels. Assign explicitly with `loc` when updating selected rows; avoid chained assignment such as `weather[mask]["column"] = ...`. The daily range is max minus min, not a second temperature measurement.

The final example takes the first three days and reverses the order of the minimum-temperature Series. Subtraction still matches the same dates, giving daily ranges 2.4, 7.2 and 4.2 degrees. The labels stay attached when you slice.

In [ ]:
weather["temp_range"] = weather["temp_max"] - weather["temp_min"]
print(weather[["temp_min", "temp_max", "temp_range"]].head(3).to_string())
print("Largest daily range:", weather["temp_range"].max())
# Keep the dates when slicing: arithmetic matches dates, not row positions.
first_high = weather.loc["2006-01-01":"2006-01-03", "temp_max"]
first_low = weather.loc["2006-01-01":"2006-01-03", "temp_min"]
print("Daily ranges, with the minimum rows reversed:")
print((first_high - first_low.iloc[::-1]).to_string())


### Check the data before summarising
The downloaded extract has no missing values or duplicate dates. A missing date is different from a present row with a missing value, so check both. NaN means unknown: replacing missing values with chosen numbers changes the data itself and can affect the conclusions. In the tiny demonstration copy below, a missing value propagates to calculations and is skipped by mean by default; that changes the number of observations contributing.

In [ ]:
print("Missing values:", weather.isna().sum().to_dict())
print("Duplicate dates:", weather.index.duplicated().sum())
expected_dates = pd.date_range(weather.index.min(), weather.index.max(), freq="D")
print("Absent dates:", len(expected_dates.difference(weather.index)))
demo = weather[["temp_mean"]].head(3).copy()
demo.iloc[1, 0] = np.nan
print("Demonstration only:")
print(demo.to_string())
print("Mean of available values:", demo["temp_mean"].mean())
print("Values used:", demo["temp_mean"].count())

### Summary statistics
`describe()` gives the non-missing count, mean, sample standard deviation (`std`), minimum, quartiles and maximum for each numeric column. The 50% row is the median. Here we select temperature and rainfall and round the displayed summary to two decimal places.

Compare mean and median rainfall: many days have little or no rain, while a few days are very wet. These summaries ignore the ordering of days, so we next group by month and examine change through time.

In [ ]:
print(weather[["temp_mean", "rain_mm"]].describe().round(2).to_string())


## 3. Weather plots
### Plot inputs
Reuse the loaded table. Each position in the date Series and temperature arrays describes the same day. `.to_numpy()` extracts numerical array values. `.dt.year` and `.dt.month` supply the calendar labels used by the masks and groupings below.

In [ ]:
dates = data["date"]
mean_temp = data["temp_mean"].to_numpy()
min_temp = data["temp_min"].to_numpy()
max_temp = data["temp_max"].to_numpy()
rain = data["rain_mm"].to_numpy()
years = dates.dt.year.to_numpy()
months = dates.dt.month.to_numpy()
print("Days:", len(data))
print("First date-temperature pair:", dates.iloc[0], mean_temp[0])


### A consistent plotting style
`fig` is the whole Figure; `ax` is one Axes object (a plotting panel, including its two axes). Methods such as `ax.plot` act on that object. A helper adds labels, a light grid and source credit; it does not transform the data. Notebook plots appear below their cells. In a script, `plt.show()` opens the figure window.

In [ ]:
plt.rcParams.update({"font.size": 12, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.figsize": (9, 3.3),
                     "axes.prop_cycle": plt.cycler(color=["#007D8A", "#D55E00", "#3F5C9A", "#9B4A7C"])})

def finish(fig, ax, title, xlabel, ylabel):
    ax.set_title(title, loc="left", fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
    fig.text(0.01, 0.01, "Odense area | ERA5 via Open-Meteo | 2006-2025 | CC BY 4.0", fontsize=8, color="#5B6773")
    fig.tight_layout(rect=[0, 0.04, 1, 1])

### Line plot: follow change through time
The dates are already sorted. Connecting successive days makes sense for this time series. A line between unrelated categories would imply an order that may not exist.

In [ ]:
fig, ax = plt.subplots()
ax.plot(dates, mean_temp, linewidth=0.5)
finish(fig, ax, "Twenty years of daily temperature", "Date", "Daily mean temperature (°C)")
plt.show()

### Zoom in
The mask selects 2024 without changing the source arrays. The full record shows repeated annual cycles; the close-up makes individual warm and cold spells visible. Neither plot alone establishes a long-term climate trend.

In [ ]:
in_2024 = years == 2024
fig, ax = plt.subplots()
ax.plot(dates[in_2024], mean_temp[in_2024], linewidth=1.2)
ax.axhline(0, color="#64717B", linestyle="--", linewidth=0.8)
finish(fig, ax, "One year: 2024", "Date", "Daily mean temperature (°C)")
plt.show()

### Several lines and a legend
For a readable comparison, first calculate the average daily minimum, mean and maximum in each month of 2024. The minimum curve is **not the coldest value in each month**: it is the average of that month's daily minima. All three lines use the same units and dates.

In [ ]:
month_numbers = np.arange(1, 13)
low_2024 = np.array([min_temp[in_2024 & (months == m)].mean() for m in month_numbers])
mean_2024 = np.array([mean_temp[in_2024 & (months == m)].mean() for m in month_numbers])
high_2024 = np.array([max_temp[in_2024 & (months == m)].mean() for m in month_numbers])
fig, ax = plt.subplots()
ax.plot(month_numbers, low_2024, label="Average daily minimum", marker="o")
ax.plot(month_numbers, mean_2024, label="Average daily mean", marker="o")
ax.plot(month_numbers, high_2024, label="Average daily maximum", marker="o")
ax.set_xticks(month_numbers)
ax.legend(fontsize=9)
finish(fig, ax, "Three views of the same year", "Month in 2024", "Temperature (°C)")
plt.show()

### Scatter: minimum and maximum
Each point pairs one day's minimum and maximum temperature. Compare January and July 2024 with two labelled colours. The winter and summer groups are separated in this sample: July days generally have higher minima and maxima. These are groups defined by the month, not clusters discovered by an algorithm. Other years or months may overlap. The same scatter method can colour points by numerical values using `c=months[...]` and a colourbar; here a two-entry legend makes the comparison simpler.

In [ ]:
winter_summer = in_2024 & ((months == 1) | (months == 7))
fig, ax = plt.subplots()
for month, name, colour in [(1, "January", "#2878B5"),
                            (7, "July", "#D65F20")]:
    selected = winter_summer & (months == month)
    ax.scatter(min_temp[selected], max_temp[selected],
               color=colour, label=name, s=35, alpha=0.75)
ax.legend(title="2024")
finish(fig, ax, "Do winter and summer days form separate groups?", "Daily minimum (°C)", "Daily maximum (°C)")
plt.show()

### Histogram: distribution
A histogram groups numerical values into intervals (bins) and counts observations in each interval. The order of days is discarded. Change the number of bins and see whether the scientific conclusion is stable.

In [ ]:
fig, ax = plt.subplots()
ax.hist(mean_temp, bins=30, edgecolor="white")
finish(fig, ax, "Which daily temperatures are common?", "Daily mean temperature (°C)", "Number of days")
plt.show()

### Bar chart: categories
Here the categories are the twelve calendar months. Each height is the average of all daily mean temperatures in that month over 2006–2025. Bars compare category values; they do not count bins as a histogram does. The zero baseline makes bar lengths interpretable.

In [ ]:
month_profile = np.array([mean_temp[months == m].mean() for m in month_numbers])
fig, ax = plt.subplots()
ax.bar(month_numbers, month_profile, color="#007D8A")
ax.set_xticks(month_numbers)
ax.set_ylim(bottom=0)
finish(fig, ax, "The annual temperature cycle", "Calendar month", "Mean daily temperature (°C)")
plt.show()

### Distribution within a season
Winter is December–February, spring March–May, summer June–August, autumn September–November. The box shows the middle 50%, its line is the median, and the whiskers reach the most extreme data points within 1.5 interquartile ranges of the box. Points beyond the whiskers are hidden in this view, not removed from the data.

In [ ]:
season_names = ["Winter", "Spring", "Summer", "Autumn"]
season_months = [(12, 1, 2), (3, 4, 5), (6, 7, 8), (9, 10, 11)]
season_values = [mean_temp[np.isin(months, group)] for group in season_months]
fig, ax = plt.subplots()
ax.boxplot(season_values, showfliers=False)
ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels(season_names)
finish(fig, ax, "How variable is each season?", "Season", "Daily mean temperature (°C)")
plt.show()

### Subplots: fair comparisons
Use the same bins and axes when comparing seasonal histograms. The two panels contain all winter and all summer days in the record; they need not have exactly equal sample sizes. Use density=True if comparing shapes with substantially different sample sizes.

In [ ]:
fig, axs = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(9, 3.6))
bins = np.linspace(-15, 30, 24)
for ax, label, values in zip(axs, ["Winter", "Summer"], [season_values[0], season_values[2]]):
    ax.hist(values, bins=bins, edgecolor="white")
    ax.set_title(label)
    ax.set_xlabel("Daily mean temperature (°C)")
    ax.grid(axis="y", alpha=0.2)
axs[0].set_ylabel("Number of days")
fig.text(0.01, 0.01, "Odense area | ERA5 via Open-Meteo | 2006-2025 | CC BY 4.0", fontsize=8)
fig.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

### Heatmap: compare many years
We build a 20×12 array of monthly means using NumPy, then display one colour per year/month. A colour bar gives the numerical scale; colour itself has no units. `origin="lower"` places the first row (2006) at the bottom.

In [ ]:
unique_years = np.arange(2006, 2026)
monthly_grid = np.array([[mean_temp[(years == y) & (months == m)].mean()
                         for m in month_numbers] for y in unique_years])
fig, ax = plt.subplots(figsize=(9, 4.5))
image = ax.imshow(monthly_grid, aspect="auto", origin="lower", cmap="coolwarm",
                  extent=[0.5, 12.5, 2005.5, 2025.5], vmin=-5, vmax=22)
ax.set_xticks(month_numbers)
ax.set_yticks([2006, 2010, 2015, 2020, 2025])
fig.colorbar(image, ax=ax, label="Monthly mean temperature (°C)")
finish(fig, ax, "Seasonality across twenty years", "Calendar month", "Year")
plt.show()

### Annual means
Averaging over a full year reduces the seasonal cycle but retains year-to-year variation. This short record at one location is not sufficient to attribute climate change. The dashed line is the mean of these twenty annual means, not a fitted warming trend.

In [ ]:
annual_values = np.array([mean_temp[years == y].mean() for y in unique_years])
fig, ax = plt.subplots()
ax.plot(unique_years, annual_values, marker="o")
ax.axhline(annual_values.mean(), linestyle="--", color="#64717B", label="Mean of annual means")
ax.legend()
ax.set_xticks([2006, 2010, 2015, 2020, 2025])
finish(fig, ax, "Which years were warmer in this sample?", "Year", "Annual mean temperature (°C)")
plt.show()

### Axes can change the impression
Both panels display exactly the same July averages. The truncated bar axis exaggerates differences in bar lengths. A line plot may legitimately zoom its y-axis when the limits are clearly labelled. Bar charts usually need a zero baseline because length represents magnitude.

In [ ]:
july = monthly_grid[:, 6]
fig, axs = plt.subplots(1, 2, figsize=(9, 3.6), sharex=True)
for ax in axs:
    ax.bar(unique_years, july, color="#007D8A")
    ax.set_xlabel("Year")
    ax.set_ylabel("July mean (°C)")
    ax.grid(axis="y", alpha=0.2)
axs[0].set_ylim(july.min() - 0.2, july.max() + 0.5)
axs[0].set_title("Baseline near the smallest value")
axs[1].set_ylim(0, july.max() + 0.5)
axs[1].set_title("Baseline at zero")
fig.text(0.01, 0.01, "Odense area | ERA5 via Open-Meteo | 2006-2025 | CC BY 4.0", fontsize=8)
fig.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

### Labels and saving
A good figure states the variables, units, time period and meaning of each colour or line. Use `fig.savefig("my-figure.png", dpi=150, bbox_inches="tight")` to save a figure before closing it.

### Grouping and resampling answer different questions
`groupby` collects rows with the same label, then applies a summary such as `mean()`, `sum()` or `count()` to each group. The label could be a city, a year or a month number. It need not describe time.

Here `weather.index.month` assigns each row a number from 1 to 12. `groupby(weather.index.month).mean()` combines January days from **all years** into one January mean, and similarly for the other months. The left plot has **12 points**: a typical annual pattern across the dataset.

`resample("MS").mean()` instead divides the date index into **consecutive calendar months** and averages the daily temperatures within each interval. January 2006 and January 2007 stay separate. The right plot has **240 points**: January 2006, February 2006, ..., December 2025. Its repeated rises and falls show successive summers and winters.

`MS` means month-start labels: the January average is labelled 1 January, but uses all available January days, not just the first day. We select the numeric temperature column before averaging. Each month averages its own daily values.

In [ ]:
profile = weather["temp_mean"].groupby(weather.index.month).mean()
monthly = weather["temp_mean"].resample("MS").mean()
print("Grouped by calendar month:", len(profile), "values")
print(profile.head(3).to_string())
print("Consecutive monthly means:", len(monthly), "values")
print(monthly.head(3).to_string())
fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))
axs[0].plot(profile.index, profile.values, marker="o")
axs[0].set_title("groupby: twelve calendar months")
axs[0].set_xlabel("Calendar month")
axs[1].plot(monthly.index, monthly.values, linewidth=1)
axs[1].set_title("resample: 240 consecutive months")
axs[1].set_xlabel("Date")
for ax in axs:
    ax.set_ylabel("Mean temperature (°C)")
    ax.grid(axis="y", alpha=0.2)
fig.text(0.01, 0.01, "Odense area | ERA5 via Open-Meteo | 2006-2025 | CC BY 4.0", fontsize=8)
fig.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

### Rolling average
Each point averages the current month and preceding eleven monthly means. `min_periods=12` leaves the first eleven results missing instead of averaging shorter windows. This smooths short-term variation; it is not a forecast. These are equally weighted monthly means, not a day-weighted annual mean. A centred rolling window would use future data and is inappropriate as a real-time forecasting input.

In [ ]:
smooth = monthly.rolling(12, min_periods=12).mean()
fig, ax = plt.subplots()
monthly.plot(ax=ax, color="#B7C5C8", label="Monthly mean", linewidth=0.9)
smooth.plot(ax=ax, color="#D55E00", label="Trailing 12-month mean", linewidth=2)
ax.legend()
finish(fig, ax, "Separate the annual cycle from slower variation", "Date", "Temperature (°C)")
plt.show()

## 4. Forecasting daily mean temperatures
### Fit the past, predict the next ten days
A forecast is a fitted rule applied beyond the observations used to estimate it. We return to **daily** mean temperatures: each predicted value is the mean temperature of one whole day, not a monthly average.

The holdout is fixed in advance as the final ten dates in the CSV, 22–31 December 2025. The preceding 365 days, 22 December 2024–21 December 2025, form the training year. We do not choose the cutoff or model order by looking for good test scores. The whole ten-day forecast is made at the end of 21 December, with no updates from later true values.


In [ ]:
daily = weather["temp_mean"].asfreq("D")
horizon = 10
training_days = 365
test = daily.iloc[-horizon:]
train = daily.iloc[-(training_days + horizon):-horizon]
print("Training:", len(train), "days from", train.index[0].date(), "to", train.index[-1].date())
print("Test:", len(test), "days from", test.index[0].date(), "to", test.index[-1].date())
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(train.index, train.values, linewidth=0.8, label="365 observed training days")
ax.axvspan(test.index[0], test.index[-1], color="#D56A35", alpha=0.2, label="Next 10 days: held out")
ax.set_xlim(train.index[0], test.index[-1])
ax.legend(fontsize=9)
finish(fig, ax, "One training year, then a ten-day forecast", "Date", "Daily mean temperature (°C)")
plt.show()


### Simple starting points
Before using a model, compare with two easily understood rules: predict the last week's average for every future day, or repeat the last observed temperature. Neither rule uses the actual temperatures in the holdout.

The recent-week average is a rolling-average idea used as a fixed forecast here. It is different from the moving-average term of ARMA, which uses earlier model errors.


In [ ]:
baseline = pd.Series(train.iloc[-7:].mean(), index=test.index)
persistence = pd.Series(train.iloc[-1], index=test.index)
print("Last week's average:", round(float(baseline.iloc[0]), 2))
print("Last observed day:", float(persistence.iloc[0]))
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(train.index[-20:], train.iloc[-20:], color="#64717B", label="Recent training observations")
ax.plot(test.index, test, "o-", color="#203040", label="Held-out actual temperatures")
ax.plot(test.index, baseline, "--", color="#D56A35", label="Last 7-day average")
ax.plot(test.index, persistence, ":", color="#8B5A9F", label="Last observed day")
ax.axvline(train.index[-1], color="#64717B", linestyle="--", linewidth=0.8)
ax.legend(fontsize=8)
finish(fig, ax, "Two forecasts that need no fitted model", "Date", "Daily mean temperature (°C)")
fig.autofmt_xdate()
plt.show()


### Model components: a yearly pattern and short-term departures
Write temperature as $T_t=s_t+r_t$: a smooth seasonal level plus a departure. A whole training year includes winter and summer, so first fit a smooth repeating yearly curve using training data only. This prevents the short-term model from being asked to explain the whole annual cycle.

For transparency, the curve below is $s_t=c_0+c_1\sin(2\pi t/365.25)+c_2\cos(2\pi t/365.25)$. The sine/cosine columns describe a yearly cycle. `np.linalg.lstsq` chooses the three coefficients to minimise squared training errors; students need not implement this solver. There is no fitted long-term trend from just one year.

After subtracting the curve, ARMA(1,1) fits $r_t=a r_{t-1}+b e_{t-1}+e_t$. It uses yesterday's departure (AR) and yesterday's model surprise (MA), plus a new unpredictable surprise. An error here means the observed departure minus its one-day-ahead prediction, not the error of a ten-day forecast.


In [ ]:
origin = train.index[0]

def seasonal_basis(index, first_day):
    day_numbers = np.asarray((index - first_day).days, dtype=float)
    angles = 2 * np.pi * day_numbers / 365.25
    return np.column_stack([np.ones(len(index)), np.sin(angles), np.cos(angles)])

training_basis = seasonal_basis(train.index, origin)
season_coefficients = np.linalg.lstsq(training_basis, train.values, rcond=None)[0]
seasonal_train = pd.Series(np.dot(training_basis, season_coefficients), index=train.index)
seasonal_future = pd.Series(np.dot(seasonal_basis(test.index, origin), season_coefficients), index=test.index)
train_residual = train - seasonal_train
fig, ax = plt.subplots(figsize=(9, 3.3))
ax.plot(train.index, train, color="#B4C4C6", linewidth=0.8, label="Observed daily mean")
ax.plot(train.index, seasonal_train, color="#D56A35", linewidth=2, label="Fitted yearly background")
ax.legend(fontsize=9)
finish(fig, ax, "A smooth yearly pattern leaves day-to-day departures", "Date", "Daily mean temperature (°C)")
plt.show()


### Fit the short-term rule and forecast
The assumptions are that the seasonal cycle is a useful background for the next ten days, and that after removing it the short-term relationship and error variability stay reasonably stable. New surprises have mean zero and are not predictable from past information. This is a teaching model, not a numerical weather forecast.

Statsmodels implements ARMA(1,1) as `ARIMA(..., order=(1, 0, 1))`: the zero means no differencing of the departures. `trend="n"` avoids adding another intercept to the already-centred departures. The fitted coefficients are chosen from the training data only. No tuning is done on the holdout.

For tomorrow, predict `a * today's departure + b * today's estimated surprise`; then add tomorrow's seasonal level. For later days, the model uses its earlier predicted departures recursively. Unknown future surprises have expected value zero. Actual test temperatures are not fed back into this ten-step forecast.


In [ ]:
from statsmodels.tsa.arima.model import ARIMA
fit = ARIMA(train_residual, order=(1, 0, 1), trend="n").fit()
result = fit.get_forecast(steps=horizon)
prediction = seasonal_future + result.predicted_mean
residual_interval = result.conf_int(alpha=0.05)
bounds = pd.DataFrame({"lower": seasonal_future + residual_interval.iloc[:, 0],
                       "upper": seasonal_future + residual_interval.iloc[:, 1]}, index=test.index)
print("Fitted AR coefficient:", round(float(fit.arparams[0]), 3))
print("Fitted MA coefficient:", round(float(fit.maparams[0]), 3))
print(pd.DataFrame({"actual": test, "forecast": prediction}).round(2).to_string())


### The ten-day forecast
The band is a model-based approximate 95% prediction interval, conditional on the fitted seasonal curve and ARMA parameters. It omits uncertainty from estimating that curve and the coefficients; it should not be presented as a calibrated weather-service interval. The new surprises cannot be known in advance. Forecasts therefore become less certain with horizon, but realised errors need not increase every single day.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(train.index[-15:], train.iloc[-15:], color="#64717B", label="Recent training observations")
ax.fill_between(test.index, bounds["lower"].values, bounds["upper"].values,
                 color="#007D8A", alpha=0.18, label="Approx. 95% prediction interval")
ax.plot(test.index, prediction, "o-", color="#007D8A", label="Seasonal curve + ARMA")
ax.plot(test.index, test, "o-", color="#203040", label="Held-out actual temperatures")
ax.plot(test.index, baseline, "--", color="#D56A35", label="Last 7-day average")
ax.axvline(train.index[-1], color="#64717B", linestyle="--", linewidth=0.8)
ax.legend(fontsize=8, ncol=2)
finish(fig, ax, "Ten daily means predicted from one fixed cutoff", "Date", "Daily mean temperature (°C)")
fig.autofmt_xdate()
plt.show()


### Errors and a fair comparison
Use error = observed − predicted: positive means the prediction was too cold. MAE is the average absolute miss; RMSE gives larger misses more weight. Every method below uses exactly the same training cutoff and ten test dates.

Including the seasonal curve alone shows whether the ARMA correction adds value, rather than attributing all improvement over an average to ARMA. On this fixed episode the combined model improves on the recent-week average and the last observed day, but is slightly worse than the seasonal curve alone. A more complex model does not automatically win, even when its training fit is good.


In [ ]:
errors = test - prediction
baseline_errors = test - baseline
methods = {"Last 7-day average": baseline,
           "Last observed day": persistence,
           "Seasonal curve": seasonal_future,
           "Seasonal curve + ARMA": prediction}
score_rows = []
for name, values in methods.items():
    misses = test - values
    score_rows.append([name, misses.abs().mean(), np.sqrt((misses ** 2).mean())])
scores = pd.DataFrame(score_rows, columns=["Model", "MAE (C)", "RMSE (C)"]).set_index("Model")
print(scores.round(3).to_string())
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(test.index, baseline_errors, "o--", color="#D56A35", label="Last 7-day average error")
ax.plot(test.index, errors, "o-", color="#007D8A", label="Seasonal curve + ARMA error")
ax.axhline(0, color="#203040", linewidth=1)
ax.legend(fontsize=9)
finish(fig, ax, "Observed minus predicted, day by day", "Held-out date", "Signed error (°C)")
fig.autofmt_xdate()
plt.show()


### Beyond the data cutoff
Only after the held-out experiment is complete do we refit the same fixed recipe on the last 365 available days, through 31 December 2025. The next ten forecasts are for 1–10 January 2026, relative to that historical cutoff. They are not a live forecast issued at lecture time, and this CSV contains no realised temperatures for scoring them.

The code repeats the same fit; it does not change model order after seeing the holdout scores.


In [ ]:
final_train = daily.iloc[-training_days:]
future_dates = pd.date_range(final_train.index[-1] + pd.Timedelta(days=1), periods=horizon, freq="D")
final_origin = final_train.index[0]
final_basis = seasonal_basis(final_train.index, final_origin)
final_coefficients = np.linalg.lstsq(final_basis, final_train.values, rcond=None)[0]
final_season = pd.Series(np.dot(final_basis, final_coefficients), index=final_train.index)
future_season = pd.Series(np.dot(seasonal_basis(future_dates, final_origin), final_coefficients), index=future_dates)
final_fit = ARIMA(final_train - final_season, order=(1, 0, 1), trend="n").fit()
future_result = final_fit.get_forecast(steps=horizon)
future_prediction = future_season + future_result.predicted_mean
future_interval = future_result.conf_int()
future_lower = future_season + future_interval.iloc[:, 0]
future_upper = future_season + future_interval.iloc[:, 1]
print("Historical data cutoff:", final_train.index[-1].date())
print("Next ten daily forecasts:")
print(future_prediction.round(2).to_string())
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(final_train.index[-20:], final_train.iloc[-20:], color="#64717B", label="Last observed daily means")
ax.fill_between(future_dates, future_lower.values, future_upper.values, color="#007D8A", alpha=0.18)
ax.plot(future_dates, future_prediction, "o-", color="#007D8A", label="Next 10 days from cutoff")
ax.axvline(final_train.index[-1], color="#64717B", linestyle="--", linewidth=0.8)
ax.legend(fontsize=9)
finish(fig, ax, "Beyond the CSV: no realised future errors available", "Date", "Daily mean temperature (°C)")
fig.autofmt_xdate()
plt.show()


## Your investigation
- Choose another forecast cutoff in advance, reserve the following ten days, and use only its preceding 365 days for fitting.
- Compare with the recent-week average, last observed day and seasonal-only curve. Does the ranking change?
- Use several cutoffs to assess robustness; do not select only the most favourable episode or repeatedly tune on the same test days.
- State one conclusion and one limitation for every plot. Forecasting is an experiment whose assumptions and errors must be checked.
